# aw_04_a2 — Stage A2: PlayWorld DPO on verifier-mined pairs (Track A, RQ2)

**Protocol**: §5.1 A2, §5.2 E-RANDPAIR control (mined later at equal pair count).  
**Parent**: A1 adapter `20260801-030335--a1-playworld-sft--s42--e24d72` (sha256 pinned in the recipe — lineage-verified at load).

Pipeline: sample K=8 candidates/prompt from the A1 policy (temperature 0.8, canonical
conditioning incl. opener seed) → score with the frozen hybrid verifier →
margin-gated chosen/rejected mining (`hybrid_verifier_rank`, margin ≥ 0.10) →
DPO (LR 5e-7, 1 epoch) → eval on the frozen suites → paired analysis A2 vs A1.

**Note**: A2 uses a NEW artifacts repo `m97j/aw-runs-a2` (one training run per repo);
eval runs still nest under `runs/` inside it.


In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt


Cloning into 'axiom-world'...
remote: Enumerating objects: 435, done.
remote: Counting objects: 100% (435/435), done.
remote: Compressing objects: 100% (275/275), done.
remote: Total 435 (delta 218), reused 348 (delta 131), pack-reused 0 (from 0)
Receiving objects: 100% (435/435), 160.21 KiB | 1.21 MiB/s, done.
Resolving deltas: 100% (218/218), done.
/content/axiom-world
Obtaining file:///content/axiom-world
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 63.8 MB/s eta 0:00:00
  Building editable for axiom-world (pyproject.toml) ... done


In [ ]:
# @title a_a2_data — mine verifier-guided preference pairs from the A1 policy
!python scripts/build_training_data.py
!python scripts/build_eval_suites.py --episodes-per-suite 300

A1_RUN_ID = "20260801-030335--a1-playworld-sft--s42--e24d72"
out = !python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1_RUN_ID}
print("\n".join(out))
adapter_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/mine_playworld_pairs.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {adapter_dir} \
  --prompt-file data/train/playworld_prompts.jsonl \
  --num-candidates 8 --temperature 0.8 --batch-size 100 \
  --selection-method hybrid_verifier_rank --minimum-margin 0.10 \
  --output data/train/playworld_preference.jsonl \
  --hf-sync-repo m97j/aw-playworld


{
  "seed": 1042,
  "sft_records": 2000,
  "prompt_records": 2000,
  "unsolvable_dropped": 0,
  "sft_fingerprint": "sha256:181cbfed0987987f467922cd1db40cb9d976374d78a67c6ab135de87f03a015f",
  "prompt_fingerprint": "sha256:cc2aef0df4f5efda3db7ccfd43c76e40260935e9b7b55ee5760490e6a4304f14",
  "train_families": [
    "train-fam0",
    "train-fam1",
    "train-fam2",
    "train-fam3",
    "train-fam4"
  ],
  "eval_families_checked": [
    "eval_adversarial-fam0",
    "eval_comp_ood-fam0",
    "eval_comp_ood-fam1",
    "eval_id-fam0",
    "eval_id-fam1",
    "eval_id-fam2",
    "eval_rule_ood-fam0",
    "eval_rule_ood-fam1",
    "eval_template_ood-fam0",
    "eval_template_ood-fam1"
  ]
}
eval_id: 300 episodes -> data/eval_suites/eval_id.jsonl (sha256:aceeea727d2b9eaed...)
eval_template_ood: 300 episodes -> data/eval_suites/eval_template_ood.jsonl (sha256:13580a6cbf7a4e566...)
eval_comp_ood: 300 episodes -> data/eval_suites/eval_comp_ood.jsonl (sha256:444191a244dcd77d1...)
eval_rule_ood: 300

In [ ]:
# @title b_a2_train — DPO from the A1 parent (lineage-verified)
!python scripts/run_experiment.py \
  --config configs/experiments/a2_playworld_dpo.yaml \
  --parent-adapter-dir {adapter_dir} \
  --override data.source.local_path=data/train/playworld_preference.jsonl \
  --hf-sync-repo m97j/aw-runs-a2


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
run_id: 20260803-005155--a2-playworld-dpo--s42--feba26
Loading weights: 100% 399/399 [00:01<00:00, 338.23it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Adding EOS to train dataset: 100% 230/230 [00:00<00:00, 33116.72 examples/s]
Tokenizing train dataset: 100% 230/230 [00:00<00:00, 543.29 examples/s]
Dropping fully truncated examples from train dataset: 100% 230/230 [00:00<00:00, 4228.70 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config 

In [ ]:
# @title c_a2_eval — A2 adapter, canonical profile
A2_RUN_ID = "20260803-005155--a2-playworld-dpo--s42--feba26"  # <- fill from b_a2_train output ("run_id: ...")
out = !python scripts/fetch_run.py --repo m97j/aw-runs-a2 --run-id {A2_RUN_ID}
print("\n".join(out))
a2_adapter_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {a2_adapter_dir} \
  --max-new-tokens 1024 --batch-size 100 \
  --hf-sync-repo m97j/aw-runs-a2


local artifacts reused (sha256 verified: sha256:b0be9afe5394...)
ADAPTER_DIR=runs/20260803-005155--a2-playworld-dpo--s42--feba26/artifacts/final_adapter
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Loading weights: 100% 399/399 [00:01<00:00, 336.58it/s]
generate(batched): 100% 3/3 [06:39<00:00, 133.24s/it]
eval_adversarial: pass_rate={'mean': 0.6767, 'ci95': [0.6233, 0.73]}
generate(batched): 100% 3/3 [06:58<00:00, 139.57s/it]
eval_comp_ood: pass_rate={'mean': 0.14, 'ci95': [0.1, 0.18]}
generate(batched): 100% 3/3 [06:56<00:00, 138.70s/it]
eval_id: pass_rate={'mean': 0.1867, 'ci95': [0.1433, 0.23]}
generate(batched): 100% 3/3 [

In [ ]:
# @title f_a2_analysis — A2 vs A1 (paired), the RQ2 primary readout
A2_EVAL_RUN = "20260803-011408--eval-playworld--s42--b6f315"
A1_EVAL_RUN = "20260801-063425--eval-playworld--s42--3bf440"

!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1_EVAL_RUN} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-a2 --run-id {A2_EVAL_RUN} --kind eval

!python scripts/run_analysis.py \
  --run-a runs/{A2_EVAL_RUN} --label-a a2-dpo \
  --run-b runs/{A1_EVAL_RUN} --label-b a1-sft \
  --output runs/{A2_EVAL_RUN}/analysis_vs_a1.json \
  --hf-sync-repo m97j/aw-runs-a2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0% 0/11 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/6.11k [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  91% 6.11k/6.69k [00:00<00:00, 29.4kB/s]
Reconstructing (incomplete total...):  53% 6.69k/12.7k [00:00<00:00, 29.4kB/s]

Fetching 11 files:   9% 1/11 [00:00<00:02,  4.28it/s]
Reconstructing (incomplete total...):  68% 12.7k/18.8k [00:00<00:00, 29.4kB/s]
Reconstructing (incomplete total...):   1% 18.8k/1.92M [00:00<01:04, 29.4kB/s]
Reconstructing (incomplete total...):   0% 18.8k/4.03M [00:00<02:16, 29.4kB/s]
Reconstructing (incomplete total...):   0% 18.8k/5.66M [00:00<03:12, 29.4kB/s]
Reconstructing (incomplete total...):   0% 18.8k/7.49M [00:00<04:14, 29.4kB/s]
Reconstructing (incomplete total...): 100% 7.49M/7.49M [00:00<00:00, 28.8MB/s]
Reconstructing (incomplete total...):  80% 7.49M/9.37M [00:00<00:00, 28.8MB/s]
Reconstructing (incomplete to

## Stage checklist
- [x] mining manifest: pairs_accepted / decision_counts sane (report both)
- [x] DPO run completed, artifacts on the Hub, lineage verified
- [x] A2 vs A1: legality + state-consistency deltas are the H2 readout
- [ ] ~~Next: E-RANDPAIR control at EQUAL pair count (aw_10) uses the same mining
      script with `--selection-method random_pairing`~~
